## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-06-00-53-00 +0000,wsj,Asian Currencies Mixed; Diminishing Risk Appet...,https://www.wsj.com/finance/currencies/asian-c...
1,2026-11-06-00-51-58 +0000,cbc,Mark Carney's quiet revolution rests on invest...,https://www.cbc.ca/news/politics/carney-champa...
2,2026-11-06-00-48-35 +0000,startribune,"Live: St. Croix Prep ousts top seed Esko, reac...",https://www.startribune.com/live-scores-minnes...
3,2026-11-06-00-48-29 +0000,nyt,"Stocks Erase Post-Midterm Elections Rally, Dra...",https://www.nytimes.com/2026/11/05/business/st...
4,2026-11-06-00-44-34 +0000,nyt,Newsom Commutes the Sentences of California’s ...,https://www.nytimes.com/2026/11/05/us/californ...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2691/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
122,trump,39
272,new,24
115,democrats,19
291,house,17
56,first,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
244,2026-11-05-17-20-00 +0000,wsj,President Trump put three justices on the Supr...,https://www.wsj.com/politics/policy/midterm-ro...,133
346,2026-11-05-11-00-00 +0000,latimes,"Hiltzik: With Trump and GOP on the run, Democr...",https://www.latimes.com/business/story/2026-11...,123
172,2026-11-05-20-47-17 +0000,nypost,"Tech bosses shouldn’t turn on Trump, he’s the ...",https://nypost.com/2026/11/05/business/tech-bo...,119
89,2026-11-05-22-26-00 +0000,wsj,The Trump administration warned conservative g...,https://www.wsj.com/politics/policy/trump-admi...,116
241,2026-11-05-17-23-49 +0000,bbc,Farage says he is worried Trump will step up I...,https://www.bbc.co.uk/news/articles/cqz0vz55jp...,112


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
244,133,2026-11-05-17-20-00 +0000,wsj,President Trump put three justices on the Supr...,https://www.wsj.com/politics/policy/midterm-ro...
346,84,2026-11-05-11-00-00 +0000,latimes,"Hiltzik: With Trump and GOP on the run, Democr...",https://www.latimes.com/business/story/2026-11...
89,77,2026-11-05-22-26-00 +0000,wsj,The Trump administration warned conservative g...,https://www.wsj.com/politics/policy/trump-admi...
302,68,2026-11-05-13-26-00 +0000,wsj,The Bank of England raised its key interest ra...,https://www.wsj.com/economy/central-banking/ba...
148,68,2026-11-05-21-10-00 +0000,cbc,California Gov. Gavin Newsom says he'll commut...,https://www.cbc.ca/news/world/california-newso...
172,68,2026-11-05-20-47-17 +0000,nypost,"Tech bosses shouldn’t turn on Trump, he’s the ...",https://nypost.com/2026/11/05/business/tech-bo...
120,64,2026-11-05-21-45-00 +0000,wsj,Democratic Socialist Nithya Raman Wins Los Ang...,https://www.wsj.com/politics/elections/democra...
243,59,2026-11-05-17-20-00 +0000,wsj,Senate Minority Leader Chuck Schumer was jubil...,https://www.wsj.com/politics/policy/chuck-schu...
189,54,2026-11-05-20-00-00 +0000,wsj,Your House vs. the Stock Market: Which Has Bee...,https://www.wsj.com/finance/investing/home-vs-...
144,49,2026-11-05-21-13-53 +0000,chitri,A brutal sell-off for AI stocks sends Wall Str...,https://www.chicagotribune.com/2026/11/05/ai-s...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
